<h1 style="font-family: 'Times New Roman';text-align: center;">Auto RC CDIO</h1>

<h2 style="font-family: 'Times New Roman';">1. Chasis (Estructural)</h2>

<div style="font-family: 'Times New Roman'; text-align: justify;">
Para un calculo de primer orden, se calcula la carga física que el motor deberá mover.

Inputs: Masa de los componentes (impresiones 3D, batería, motor). Masa de la carga útil (TBD kg).

Process: Suma lineal de las masas: 
$$M_{total} = M_{chasis} + M_{electronica} + M_{mecanica} + M_{carga}$$
Outputs:
Masa total ($M_{total}$) en kg. (Este será el input crítico para Neumáticos).
</div>

In [6]:
# 1. SUBSISTEMA CHASIS (Estructural y Masas)

class Chasis:
    def __init__(self, masa_chasis, masa_electronica, masa_mecanica, masa_carga_util):
        # INPUTS
        self.m_chasis = masa_chasis
        self.m_elec = masa_electronica
        self.m_mec = masa_mecanica
        self.m_carga = masa_carga_util
        
    def calcular_masa_total(self):
        # PROCESS
        masa_total = self.m_chasis + self.m_elec + self.m_mec + self.m_carga
        # OUTPUT
        return masa_total

<h2 style="font-family: 'Times New Roman';">2. Electrónica (Control y Energía)</h2>

<div style="font-family: 'Times New Roman'; text-align: justify;">
Aquí definimos cuánta energía eléctrica tenemos y a qué ritmo la entregamos.

Inputs:Voltaje nominal de la batería ($V_{bat}$).Capacidad de la batería en Amperios-hora ($Ah$).Consumo de corriente nominal del motor ($I_{nom}$).

Process: Potencia eléctrica disponible: $P_{elec} = V_{bat} \cdot I_{nom}$

Tiempo teórico de autonomía: $t_{max} = Ah / I_{nom}$

Outputs: Potencia eléctrica ($P_{elec}$) en Watts.Voltaje disponible ($V_{bat}$) en Voltios.
</div>

In [7]:
# 2. SUBSISTEMA ELECTRÓNICA (Energía)

class Electronica:
    def __init__(self, voltaje_nominal, capacidad_ah, corriente_consumo):
        # INPUTS
        self.v_bat = voltaje_nominal
        self.capacidad = capacidad_ah
        self.i_consumo = corriente_consumo
        
    def procesar_energia(self):
        # PROCESS
        potencia_elec = self.v_bat * self.i_consumo
        autonomia_horas = self.capacidad / self.i_consumo
        # OUTPUTS
        return self.v_bat, potencia_elec, autonomia_horas

<h2 style="font-family: 'Times New Roman';">3. Mecánica (Actuación y Transmisión)</h2> 

<div style="font-family: 'Times New Roman'; text-align: justify;">
Aquí modelamos la transformación de la energía eléctrica a mecánica y su reducción a través de la correa y los engranajes cónicos.

Inputs: Voltaje efectivo ($V_{eff}$) desde Control. 
Constante de velocidad del motor ($K_v$ en RPM/V).Relación de reducción total ($N_{red}$ = ratio polea $\cdot$ ratio diferencial).
Eficiencia mecánica del sistema ($\eta$, un valor de 0 a 1).

Process:Velocidad del motor: $$RPM_{motor} = V_{eff} \cdot K_v$$
Velocidad a la salida del eje: $$RPM_{eje} = RPM_{motor} / N_{red}$$
Torque a la salida del eje (asumiendo potencia constante teórica): $$T_{eje} = (T_{motor} \cdot N_{red}) \cdot \eta$$

Outputs:Velocidad angular del eje ($RPM_{eje}$).
Torque disponible en el eje ($T_{eje}$).
</div>

In [ ]:
# 3. SUBSISTEMA MECÁNICA (Motor y Transmisión)

class Mecanica:
    def __init__(self, kv_motor, relacion_reduccion, eficiencia_sistema, torque_motor_nm):
        # INPUTS
        self.kv = kv_motor
        self.n_red = relacion_reduccion
        self.eficiencia = eficiencia_sistema
        self.t_motor = torque_motor_nm
        
    def calcular_cinematica_y_torque(self, voltaje_efectivo):
        # PROCESS
        rpm_motor = voltaje_efectivo * self.kv
        rpm_eje = rpm_motor / self.n_red
        
        # En primer orden, el torque se multiplica por la reducción y se multiplica por la eficiencia
        torque_eje = (self.t_motor * self.n_red) * self.eficiencia
        
        # OUTPUTS
        return rpm_eje, torque_eje

<h2 style="font-family: 'Times New Roman';">4. Suspensión y Neumáticos</h2>

<div style="font-family: 'Times New Roman'; text-align: justify;">

Inputs: Velocidad angular y Torque del eje (desde Mecánica).Masa total (desde Chasis).Radio de la rueda ($r$).Gravedad ($g = 9.81 m/s^2$).Coeficiente de fricción estática del suelo ($\mu$).

Process: Fuerza normal sobre las ruedas de tracción (asumiendo peso distribuido equitativamente, si es tracción trasera, usualmente es un porcentaje, ej. 50%): $$F_n = M_{total} \cdot g \cdot 0.5$$ Fuerza límite antes de derrapar: $$F_{limite} = \mu \cdot F_n$$ Fuerza de empuje generada por el torque: $$F_{empuje} = T_{eje} / r$$ Velocidad lineal: $$V_{lineal} = RPM_{eje} \cdot (2\pi / 60) \cdot r$$

Outputs: Velocidad lineal real del auto ($V_{lineal}$) en m/s.Verificación de tracción: Una lógica simple SI (F_empuje > F_limite) entonces el auto derrapa y pierde eficiencia, si no, avanza traccionando correctamente.

</div>

In [9]:
# 4. SUBSISTEMA NEUMÁTICOS (Interacción Suelo-Rueda)

class Neumaticos:
    def __init__(self, radio_rueda, coef_friccion_estatica):
        # INPUTS
        self.radio = radio_rueda
        self.mu = coef_friccion_estatica
        self.gravedad = 9.81
        self.pi = 3.14159
        
    def calcular_traccion_y_avance(self, rpm_eje, torque_eje, masa_total):
        # PROCESS
        # 1. Fuerza normal (asumiendo 50% del peso en el eje motriz para tracción en 2 ruedas)
        fuerza_normal = masa_total * self.gravedad * 0.5
        
        # 2. Fuerza límite de fricción antes de derrapar
        fuerza_limite = self.mu * fuerza_normal
        
        # 3. Fuerza de empuje generada por el motor y la transmisión
        fuerza_empuje = torque_eje / self.radio
        
        # 4. Velocidad lineal (m/s)
        velocidad_lineal = rpm_eje * (2 * self.pi / 60) * self.radio
        
        # Verificación de deslizamiento (Check)
        derrapa = fuerza_empuje > fuerza_limite
        
        # OUTPUTS
        return velocidad_lineal, fuerza_empuje, fuerza_limite, derrapa

<h2 style="font-family: 'Times New Roman';">5. Control y Telemetría (El factor "RC")</h2>

<div style="font-family: 'Times New Roman'; text-align: justify;">
En primer orden, el control es un simple mapeo de la voluntad del usuario a la máquina.

Inputs: Aceleración del usuario en el joystick (ej. un valor $U$ de 0% a 100%).Voltaje disponible (desde Electrónica).

Process: Mapeo lineal para obtener el voltaje efectivo enviado al motor: $V_{eff} = V_{bat} \cdot (U / 100)$

Outputs: Voltaje efectivo ($V_{eff}$) enviado al motor.
</div>

In [10]:
# 5. SUBSISTEMA CONTROL (Señal de Actuación)

class Control:
    def __init__(self, aceleracion_porcentaje):
        # INPUTS
        self.aceleracion = aceleracion_porcentaje / 100.0  # Convertir % a decimal (0.0 a 1.0)
        
    def calcular_voltaje_efectivo(self, voltaje_disponible):
        # PROCESS
        voltaje_efectivo = voltaje_disponible * self.aceleracion
        # OUTPUT
        return voltaje_efectivo

<h1 style="font-family: 'Times New Roman';">Integración: Misión Nivel 0</h1>

<div style="font-family: 'Times New Roman'; text-align: justify;">
La sección de integración consolida la arquitectura jerárquica del vehículo al conectar secuencialmente las entradas y salidas de cada subsistema bajo el enfoque IPO. 
Al unificar los datos de masa, energía, control, transmisión y neumáticos, el modelo simula el rendimiento global para el cumplimiento de la misión de nivel 0. 
Este bloque es clave en el marco CDIO, ya que valida el sistema completo y facilita la transición directa del diseño digital al prototipo físico operativo.
</div>

In [11]:
# INTEGRACIÓN: Misión Nivel 0 (Input -> Process -> Output)


# --- 1. INSTANCIACIÓN DE INPUTS (Condiciones Iniciales) ---
mi_chasis = Chasis(masa_chasis=0.8, masa_electronica=0.4, masa_mecanica=0.6, masa_carga_util=0.7) # Total 2.5kg
mi_electronica = Electronica(voltaje_nominal=7.4, capacidad_ah=5.0, corriente_consumo=10.0) # Batería LiPo 2S
mi_control = Control(aceleracion_porcentaje=100) # Acelerador a fondo
mi_mecanica = Mecanica(kv_motor=1500, relacion_reduccion=3.5, eficiencia_sistema=0.85, torque_motor_nm=0.1)
mis_neumaticos = Neumaticos(radio_rueda=0.04, coef_friccion_estatica=0.8) # Ruedas de 4cm en asfalto
distancia_mision = 15.0 # Metros de A hasta B

# --- 2. PROCESAMIENTO SECUENCIAL (Conectando los bloques) ---
masa_vehiculo = mi_chasis.calcular_masa_total()

voltaje_bat, potencia, autonomia = mi_electronica.procesar_energia()

voltaje_real_motor = mi_control.calcular_voltaje_efectivo(voltaje_bat)

rpm_salida, torque_salida = mi_mecanica.calcular_cinematica_y_torque(voltaje_real_motor)

v_lineal, f_empuje, f_limite, alerta_derrape = mis_neumaticos.calcular_traccion_y_avance(rpm_salida, torque_salida, masa_vehiculo)

tiempo_mision = distancia_mision / v_lineal

# --- 3. OUTPUTS GLOBALES ---
print("--- RESULTADOS DE LA MISIÓN NIVEL 0 ---")
print(f"Masa Total a mover: {masa_vehiculo} kg")
print(f"Potencia Eléctrica Bruta: {potencia} W (Voltaje aplicado al motor: {voltaje_real_motor} V)")
print(f"Velocidad en el eje final: {rpm_salida:.2f} RPM | Torque entregado: {torque_salida:.3f} N·m")
print(f"Velocidad Lineal de Avance: {v_lineal:.2f} m/s")

print("\n--- ANÁLISIS DE TRACCIÓN ---")
print(f"Fuerza de Empuje: {f_empuje:.2f} N | Límite antes de derrapar: {f_limite:.2f} N")
if alerta_derrape:
    print("¡ADVERTENCIA! Las ruedas van a derrapar (Fuerza Empuje > Límite).")
else:
    print("Tracción óptima confirmada. No hay derrape.")

print(f"\n--- RENDIMIENTO DE LA MISIÓN ---")
print(f"Tiempo estimado para recorrer {distancia_mision}m: {tiempo_mision:.2f} segundos")

--- RESULTADOS DE LA MISIÓN NIVEL 0 ---
Masa Total a mover: 2.5 kg
Potencia Eléctrica Bruta: 74.0 W (Voltaje aplicado al motor: 7.4 V)
Velocidad en el eje final: 3171.43 RPM | Torque entregado: 0.298 N·m
Velocidad Lineal de Avance: 13.28 m/s

--- ANÁLISIS DE TRACCIÓN ---
Fuerza de Empuje: 7.44 N | Límite antes de derrapar: 9.81 N
Tracción óptima confirmada. No hay derrape.

--- RENDIMIENTO DE LA MISIÓN ---
Tiempo estimado para recorrer 15.0m: 1.13 segundos
